# T1 — Repartitioning Analysis
Sara Milovanova & Biljana Vitanova

This notebook documents:
- Schema changes discovered across all four TLC datasets
- Issues encountered during repartitioning
- Column mapping and unified schema for Yellow Taxi
- Per-year statistics (file sizes, row counts, row groups)

In [1]:
from pathlib import Path
import pyarrow.parquet as pq
import pandas as pd

SHARED_DIR   = Path("/d/hpc/projects/FRI/bigdata/data/Taxi")
DOWNLOAD_DIR = Path("/d/hpc/projects/FRI/bigdata/students/sm_bv/taxi_new")
OUTPUT_DIR   = Path("/d/hpc/projects/FRI/bigdata/students/sm_bv/final_project/data/partitioned")

DATASETS = ["yellow_tripdata", "green_tripdata", "fhv_tripdata", "fhvhv_tripdata"]

---
## 1. Schema Changes per Dataset

Detected by reading the Parquet file footer (metadata only) of every source file and comparing consecutive schemas.

### Yellow Taxi — 3 full schema generations (most complex)

| When | What changed |
|---|---|
| **2009 baseline** | Old schema: `vendor_name`, `Trip_Pickup_DateTime`, lat/lon columns (`Start_Lon/Lat`, `End_Lon/Lat`), `Rate_Code` as double |
| **2010-01** | Complete rename to lowercase: `vendor_id`, `pickup_datetime` as **string**, `rate_code` as **string** |
| **2010-02** | `rate_code` string → int64 (flips back to string in 2010-04) — **caused `pa.unify_schemas` crash** |
| **2011-01** | Full modern schema: `VendorID`, `tpep_pickup_datetime` as timestamp, `PULocationID`/`DOLocationID` replace lat/lon |
| **2011–2017** | `congestion_surcharge` / `airport_fee` / `improvement_surcharge` flip between `null` and `double` across months |
| **2018-07** | `passenger_count`, `RatecodeID` int64 → double |
| **2023-02** | `airport_fee` renamed to `Airport_fee` (capitalisation); int64 → int32 for IDs; string → large_string |
| **2025-02** | `cbd_congestion_fee` added |

### Green Taxi — stable structure, minor drift

| When | What changed |
|---|---|
| **2014 baseline** | Modern schema from the start |
| **2014–2018** | `improvement_surcharge`, `trip_type`, `congestion_surcharge` flip null↔double across months |
| **2018-07** | `RatecodeID`, `passenger_count`, `payment_type`, `trip_type` int64 → double |
| **2019-01** | `congestion_surcharge` permanently non-null |
| **2023-02** | int64 → int32 for location IDs; string → large_string |
| **2025-02** | `cbd_congestion_fee` added |

### FHV — very stable (7 columns throughout)

| When | What changed |
|---|---|
| **2015 baseline** | 7 columns, `SR_Flag` is null |
| **2017-05** | `SR_Flag` null → double |
| **2023-02** | string → large_string; `PUlocationID`/`DOlocationID` double → int64 |

### FHVHV — stable (24 columns)

| When | What changed |
|---|---|
| **2019-02 baseline** | 24 columns |
| **2019-04** | `wav_match_flag` null → string |
| **2019-07** | `airport_fee` null → double (flips back and forth until 2020-11) |
| **2023-02** | int64 → int32 for location IDs; all strings → large_string |
| **2025-02** | `cbd_congestion_fee` added |

---
## 2. Issues Encountered During T1

### Issue 1 — Dask workers dying (OOM)
The original script used `SLURMCluster` with 4 Dask workers. Despite requesting `memory="64GB"`, workers received only ~15 GB because `dask-jobqueue` divides the allocation across internal threads. Yellow taxi files pushed workers to 80% memory → pause → eventual kill.  
**Fix:** Dropped Dask entirely. Pure PyArrow + `sbatch` job.

### Issue 2 — Script hanging on login node
Running the PyArrow script directly on the HPC login node caused it to be killed by the node's resource limits.  
**Fix:** Always run via `srun` (interactive) or `sbatch` (background).

### Issue 3 — `batch.cast(unified_schema)` silently failing
PyArrow's `cast()` only converts column *types* — it cannot add or remove columns. Batches from old-schema files (different column names) silently produced no output. Only a handful of rows with dirty timestamps (years 2041, 2053) appeared in output.  
**Fix:** Replaced with `align_batch()` which rebuilds each batch column-by-column, inserting `pa.nulls()` for missing columns.

### Issue 4 — `pa.unify_schemas()` crashing on string vs int64
Yellow's `rate_code` is `string` in 2010 files and `int64` in others. `promote_options="permissive"` handles numeric promotions but has no rule for `string ↔ int64`.  
**Fix:** `safe_unify_schemas()` tries native promotion first, falls back to `large_string` for truly incompatible types.

### Issue 5 — Too many output files per year
Setting `max_rows_per_file=2_000_000` (equal to row group size) created one file per row group. Yellow 2009 produced 84 separate files.  
**Fix:** Removed `max_rows_per_file`. Now one file per year with multiple 2M-row row groups inside.

---
## 3. Yellow Taxi — Column Mapping & Unified Schema

The 47 raw columns (across all schema generations) collapse into **24 canonical columns**.

### Rename groups

| Canonical name | Type | Old names |
|---|---|---|
| `VendorID` | double | `vendor_name` (2009), `vendor_id` (2010), `VendorID` (2011+) |
| `tpep_pickup_datetime` | timestamp[us] | `Trip_Pickup_DateTime` (2009), `pickup_datetime` (2010), `tpep_pickup_datetime` (2011+) |
| `tpep_dropoff_datetime` | timestamp[us] | `Trip_Dropoff_DateTime` (2009), `dropoff_datetime` (2010), `tpep_dropoff_datetime` (2011+) |
| `passenger_count` | double | `Passenger_Count` (2009), `passenger_count` (2010+) |
| `trip_distance` | double | `Trip_Distance` (2009), `trip_distance` (2010+) |
| `RatecodeID` | double | `Rate_Code` (2009), `rate_code` (2010), `RatecodeID` (2011+) |
| `store_and_fwd_flag` | string | `store_and_forward` (2009), `store_and_fwd_flag` (2010+) |
| `payment_type` | double | `Payment_Type` (2009), `payment_type` (2010+) |
| `fare_amount` | double | `Fare_Amt` (2009), `fare_amount` (2010+) |
| `extra` | double | `surcharge` (2009), `extra` (2011+) |
| `tip_amount` | double | `Tip_Amt` (2009), `tip_amount` (2010+) |
| `tolls_amount` | double | `Tolls_Amt` (2009), `tolls_amount` (2010+) |
| `total_amount` | double | `Total_Amt` (2009), `total_amount` (2010+) |
| `airport_fee` | double | `airport_fee` + `Airport_fee` (capitalisation inconsistency fixed in 2023) |

### Present only in older files (null for 2011+)

| Column | Type | Source |
|---|---|---|
| `pickup_longitude` | double | `Start_Lon` (2009), `pickup_longitude` (2010) |
| `pickup_latitude` | double | `Start_Lat` (2009), `pickup_latitude` (2010) |
| `dropoff_longitude` | double | `End_Lon` (2009), `dropoff_longitude` (2010) |
| `dropoff_latitude` | double | `End_Lat` (2009), `dropoff_latitude` (2010) |

### Present only in newer files (null for earlier years)

| Column | Type | Introduced |
|---|---|---|
| `PULocationID` | double | 2011 — replaced lat/lon |
| `DOLocationID` | double | 2011 — replaced lat/lon |
| `mta_tax` | double | 2011 |
| `improvement_surcharge` | double | 2015 |
| `congestion_surcharge` | double | 2019 |
| `cbd_congestion_fee` | double | 2025 |

### Dropped
| Column | Reason |
|---|---|
| `__index_level_0__` | Pandas artifact, not real data |

---
## 4. Statistics — Source Files

In [2]:
rows = []
for prefix in DATASETS:
    files = sorted(
        list(SHARED_DIR.glob(f"{prefix}_*.parquet")) +
        list(DOWNLOAD_DIR.glob(f"{prefix}_*.parquet"))
    )
    total_mb = sum(f.stat().st_size for f in files) / 1e6
    rows.append({"dataset": prefix, "source_files": len(files), "total_size_MB": round(total_mb, 1)})

pd.DataFrame(rows)

,dataset,source_files,total_size_MB
0,yellow_tripdata,206,31969.4
1,green_tripdata,146,1330.3
2,fhv_tripdata,134,6089.5
3,fhvhv_tripdata,85,38263.6


---
## 5. Statistics — Partitioned Output (per dataset per year)

In [3]:
stats = []

for prefix in DATASETS:
    out_path = OUTPUT_DIR / prefix
    if not out_path.exists():
        print(f"{prefix}: output not yet available")
        continue
    for year_dir in sorted(out_path.iterdir()):
        if not year_dir.is_dir():
            continue
        files = list(year_dir.glob("*.parquet"))
        if not files:
            continue
        size_mb   = sum(f.stat().st_size for f in files) / 1e6
        total_rows = 0
        total_rg   = 0
        for f in files:
            meta = pq.read_metadata(f)
            total_rows += meta.num_rows
            total_rg   += meta.num_row_groups
        stats.append({
            "dataset":    prefix,
            "year":       year_dir.name,
            "files":      len(files),
            "size_MB":    round(size_mb, 1),
            "rows":       total_rows,
            "row_groups": total_rg,
            "avg_rg_rows": round(total_rows / total_rg) if total_rg else 0,
        })

df = pd.DataFrame(stats)
pd.set_option("display.max_rows", 200)
df

,dataset,year,files,size_MB,rows,row_groups,avg_rg_rows
0,yellow_tripdata,2001,1,0.2,27,21,1
1,yellow_tripdata,2002,1,0.6,498,80,6
2,yellow_tripdata,2003,1,0.3,50,41,1
3,yellow_tripdata,2004,1,0.0,1,1,1
4,yellow_tripdata,2007,1,0.0,1,1,1
5,yellow_tripdata,2008,1,4.1,771,571,1
6,yellow_tripdata,2010,1,9104.9,169001163,5916,28567
7,yellow_tripdata,2011,1,2532.0,176887263,6191,28572
8,yellow_tripdata,2012,1,2512.4,171359008,5996,28579
9,yellow_tripdata,2013,1,2504.5,171816340,6013,28574


---
## 6. Statistics — Summary per Dataset

In [4]:
summary = (
    df.groupby("dataset")
    .agg(
        years      = ("year", "nunique"),
        total_files= ("files", "sum"),
        total_MB   = ("size_MB", "sum"),
        total_rows = ("rows", "sum"),
        total_rg   = ("row_groups", "sum"),
    )
    .reset_index()
)
summary["total_GB"] = (summary["total_MB"] / 1024).round(2)
summary["avg_rg_rows"] = (summary["total_rows"] / summary["total_rg"]).round(0).astype(int)
summary

,dataset,years,total_files,total_MB,total_rows,total_rg,total_GB,avg_rg_rows
0,fhv_tripdata,12,12,6779.1,798450799,27964,6.62,28553
1,fhvhv_tripdata,8,8,44472.6,1521319081,53418,43.43,28480
2,green_tripdata,22,22,1848.4,84153708,3412,1.81,24664
3,yellow_tripdata,40,40,33427.4,1663188038,59237,32.64,28077


---
## 7. Verify Unified Schema of Output

In [5]:
for prefix in DATASETS:
    out_path = OUTPUT_DIR / prefix
    if not out_path.exists():
        print(f"{prefix}: not available\n")
        continue
    files = list(out_path.rglob("*.parquet"))
    if not files:
        print(f"{prefix}: no files\n")
        continue

    # Check all output files share the same schema
    schemas = {str(pq.read_schema(f)) for f in files}
    consistent = len(schemas) == 1

    ref_schema = pq.read_schema(files[0])
    print(f"{'='*55}")
    print(f"{prefix}  ({len(files)} output files)")
    print(f"Schema consistent: {consistent}")
    print(f"Columns ({len(ref_schema)}):")
    for field in ref_schema:
        print(f"  {field.name:40s} {field.type}")
    print()

yellow_tripdata  (40 output files)
Schema consistent: True
Columns (47):
  vendor_name                              string
  Trip_Pickup_DateTime                     string
  Trip_Dropoff_DateTime                    string
  Passenger_Count                          int64
  Trip_Distance                            double
  Start_Lon                                double
  Start_Lat                                double
  Rate_Code                                double
  store_and_forward                        double
  End_Lon                                  double
  End_Lat                                  double
  Payment_Type                             string
  Fare_Amt                                 double
  surcharge                                double
  mta_tax                                  double
  Tip_Amt                                  double
  Tolls_Amt                                double
  Total_Amt                                double
  vendor_id                 